# 05 — Machine-learning baseline

A predictive model for 77 K H₂ uptake from structure and conditions, done the way the manual requires (§14.4): **cross-validation grouped by paper**, so performance is measured on papers the model never trained on; honest fold-to-fold uncertainty; transparent feature attribution (SHAP); and the fold isolation verified, not assumed. All logic is in `hycan.features` and `hycan.ml`, unit-tested in `tests/test_ml.py`.

In [1]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
_d = Path.cwd()
while _d != _d.parent and not (_d / 'pyproject.toml').exists():
    _d = _d.parent
os.chdir(_d); sys.path.insert(0, str(_d / 'src'))
import numpy as np, pandas as pd
from hycan.load import load_dataset
from hycan import features, ml
df = load_dataset()
frame = features.feature_frame(df)
X, y, groups = features.design_matrix(frame)
print('modeling subset:', len(frame), 'rows,', frame['paper_id'].nunique(), 'papers')

modeling subset: 191 rows, 28 papers


## The modeling subset
§12.3-filtered, Tier A/B, 77 K, wt% target, method-confirmed BET area, `total`-uptake excluded. Features: BET area, micropore / ultramicropore / total pore volume, pressure, and material_class one-hot. **Doping is deliberately excluded** (§14.3): the doped subset is too small and too unevenly distributed across conditions and concentration units to be more than a paper identifier in disguise.

In [2]:
print('rows:', len(frame), ' papers:', frame['paper_id'].nunique())
print('tier split:', frame['reproducibility_tier'].value_counts().to_dict())
print('features:', list(X.columns))

rows: 191  papers: 28
tier split: {'B': 138, 'A': 53}
features: ['bet_surface_area_m2_g', 'micropore_volume_cm3_g', 'ultramicropore_volume_cm3_g', 'total_pore_volume_cm3_g', 'pressure_bar', 'mat_MWCNT', 'mat_SWCNT', 'mat_activated_carbon', 'mat_carbide_derived_carbon', 'mat_carbon_aerogel', 'mat_carbon_nanofiber', 'mat_composite', 'mat_doped_carbon', 'mat_graphene', 'mat_other', 'mat_reduced_graphene_oxide', 'mat_templated_carbon']


At **191 rows / 28 papers** this is nearly triple the 102 rows / 11 papers the manual's §14.1 recorded, because Phase D and the verification pass widened the 77 K BET corpus. 28 groups make a grouped-CV estimate meaningful rather than noise.

## Grouped cross-validation
Five folds, split by `paper_id`. Standard CV would leak a paper across train and test and report dishonestly optimistic numbers; grouping measures generalization to new papers. Fold isolation is checked explicitly.

In [3]:
cv = ml.cross_validate(X, y, groups)
print(f"{cv['n_splits']} folds, {cv['n_rows']} rows, {cv['n_papers']} papers\n")
for name, m in cv['models'].items():
    print(f"{name:16s} MAE {m['mae_mean']:.3f} +/- {m['mae_sd']:.3f} wt%   "
          f"R2 {m['r2_mean']:.3f} +/- {m['r2_sd']:.3f}")

5 folds, 191 rows, 28 papers

LinearRegression MAE 0.876 +/- 0.304 wt%   R2 0.415 +/- 0.495
RandomForest     MAE 0.465 +/- 0.058 wt%   R2 0.848 +/- 0.068
XGBoost          MAE 0.486 +/- 0.060 wt%   R2 0.836 +/- 0.078


**The tree models generalize well to unseen papers:** RandomForest and XGBoost both reach **R² ≈ 0.84 (MAE ≈ 0.47–0.49 wt%)**, far above the linear baseline (R² ≈ 0.4, and high fold variance — it cannot handle the sparse pore features without imputation). The two tree models are within each other's fold spread, so they are statistically indistinguishable here. XGBoost is used for the attribution and the predicted-vs-measured figure because it handles the pore-volume missingness natively, with no imputation artifact.

## Predicted vs measured (out-of-fold) — `figures/fig6_pred_vs_measured`
Every point predicted by a model that did not train on its paper.

In [4]:
from sklearn.metrics import r2_score, mean_absolute_error
pred = ml.out_of_fold_predictions(X, y, groups, 'XGBoost')
print(f'out-of-fold R2 = {r2_score(y, pred):.3f}, MAE = {mean_absolute_error(y, pred):.3f} wt%')

out-of-fold R2 = 0.867, MAE = 0.486 wt%


## Feature attribution — `figures/fig7_shap`
SHAP mean-|value| on the XGBoost model.

In [5]:
model = ml.fit_full(X, y, 'XGBoost')
importance, _ = ml.shap_summary(model, X)
print(importance.head(8).round(4).to_string())

pressure_bar                   1.0365
bet_surface_area_m2_g          0.5987
micropore_volume_cm3_g         0.1789
total_pore_volume_cm3_g        0.1326
mat_activated_carbon           0.0329
ultramicropore_volume_cm3_g    0.0125
mat_reduced_graphene_oxide     0.0104
mat_doped_carbon               0.0094


**Pressure and BET area dominate**, with pore volumes a clear second tier and material-class one-hots contributing little — exactly the 77 K physisorption picture: uptake is set by how much surface there is and how hard the gas is pushed onto it, not by which carbon family the sample belongs to. With doping excluded (§14.3), this figure is about pore structure and conditions; it is not evidence that chemistry never matters, only that structure and conditions explain most of the variance here.

## Design trade-off — `figures/fig8_pareto`
Uptake vs BET with the Pareto-optimal frontier (most uptake for the least surface area) highlighted — the database used as a prospective synthesis guide.

## Conclusions
1. A paper-grouped model predicts 77 K uptake with **R² ≈ 0.84, MAE ≈ 0.5 wt%** on held-out papers — a genuine generalization estimate, not a memorized fit.
2. **Pressure and BET surface area are the dominant predictors;** pore volumes refine, material class barely moves the prediction.
3. **Honest limitations:** 28 papers is adequate but not large; doping is excluded for want of a comparable subset; the linear baseline is poor because the pore features are sparse; and no lab samples are overlaid on Fig 6 because the Perez-lab samples are Raman/XPS-characterized and carry no H₂ uptake measurement.
4. The model and its attribution are reproducible from this notebook and the tested `hycan.ml` module.